# WWEX-1A October milestones

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

!git clone https://github.com/Break-Through-Tech/WWEX-1A-delivery-performance-and-smart-carrier-selection-for-multi-carrier-shipping.git

train = pd.read_csv('WWEX-1A-delivery-performance-and-smart-carrier-selection-for-multi-carrier-shipping/data/train.csv')
test = pd.read_csv('WWEX-1A-delivery-performance-and-smart-carrier-selection-for-multi-carrier-shipping/data/test.csv')

In [ ]:
# Load the raw data. ZIPs are read as text so we don't lose leading zeros
path = "WWEX-1A-delivery-performance-and-smart-carrier-selection-for-multi-carrier-shipping/data/"
train = pd.read_csv(path + "train.csv", dtype={"origin_zip": str, "dest_zip": str})
test = pd.read_csv(path + "test.csv", dtype={"origin_zip": str, "dest_zip": str})

# The correct spellings we want to end up with
carriers = ["Falcon Post", "Northstar Parcel", "Summit Express",
            "Ironclad Freight", "Vela Freight", "Cobalt Regional"]
services = ["Ground", "Expedited", "TwoDay", "Overnight", "LTL Standard", "LTL Guaranteed"]

# Lookup: lowercase version -> correct spelling (e.g. "falcon post" -> "Falcon Post")
carrier_map = {n.lower(): n for n in carriers}
service_map = {n.lower(): n for n in services}

# Columns that have missing or messy numbers
fill_cols = ["weight_lbs", "length_in", "width_in", "height_in"]

# Steps 1-3 run on both train and test
for df in (train, test):
    # 1. Fix messy text: remove extra spaces, make lowercase, then swap in the correct spelling
    df["carrier"] = df["carrier"].str.strip().str.lower().map(carrier_map)
    df["service_level"] = df["service_level"].str.strip().str.lower().map(service_map)

    # 2. Fix ZIPs: put back the leading zero that got lost (2108 -> 02108)
    for c in ["origin_zip", "dest_zip"]:
        df[c] = df[c].str.strip().str.zfill(5)

    # 3. Turn date text into real dates so we can sort by time later
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["ship_date"] = pd.to_datetime(df["ship_date"])

# 4. Remove duplicate rows (done after the text cleanup so near-duplicates get caught too)
train = train.drop_duplicates().reset_index(drop=True)
test = test.drop_duplicates().reset_index(drop=True)

for df in (train, test):
    # 5a. A weight of 0 isn't real, so treat it as missing
    df.loc[df["weight_lbs"] == 0, "weight_lbs"] = np.nan

    # 5b. Find weights that are way too big (these were grams typed in as pounds)
    grams = ((df["shipment_mode"] == "parcel") & (df["weight_lbs"] > 150)) | \
            ((df["shipment_mode"] == "ltl_freight") & (df["weight_lbs"] > 20000))

    # 5c. Convert those from grams to pounds
    df.loc[grams, "weight_lbs"] = df.loc[grams, "weight_lbs"] / 453.592

    # 6. Mark which values were missing BEFORE we fill them in
    for c in fill_cols:
        df[c + "_na"] = df[c].isnull()

# 7. Fill missing values with the typical (median) value for that shipment type
# Medians come from train only, so test doesn't leak into our results
meds = train.groupby("shipment_mode")[fill_cols].median()
print(meds)
for df in (train, test):
    for c in fill_cols:
        df[c] = df[c].fillna(df["shipment_mode"].map(meds[c]))

# Checks: make sure everything worked
print(len(train), len(test))                        # expect about 39,949 and 10,114
print(train[fill_cols].isnull().sum().sum(), test[fill_cols].isnull().sum().sum())   # expect 0 0
print(train.groupby("shipment_mode")[fill_cols].describe().T)   # weights should look realistic now

# Date ranges (we need these to decide how to split the data)
print("train:", train["ship_date"].min().date(), "to", train["ship_date"].max().date())
print("test: ", test["ship_date"].min().date(), "to", test["ship_date"].max().date())

In [ ]:
from sklearn.metrics import mean_absolute_error  # average size of our mistakes, in days

# sort shipments by date
train = train.sort_values("ship_date").reset_index(drop=True)

# split on a DATE, not a row number, so no calendar day lands in both train and validation
split_date = train["ship_date"].quantile(0.8, interpolation="nearest").normalize()

# older shipments for training, newest shipments for validation
tr = train[train["ship_date"] < split_date].copy()
val = train[train["ship_date"] >= split_date].copy()
assert tr["ship_date"].max() < val["ship_date"].min()   # fails loudly if a day ever appears on both sides

# check the date ranges and row counts
print("tr: ", tr["ship_date"].min().date(), "to", tr["ship_date"].max().date(), len(tr))
print("val:", val["ship_date"].min().date(), "to", val["ship_date"].max().date(), len(val))

# use promised days as the prediction and measure the average error
print("Naive MAE:", mean_absolute_error(val["transit_days"], val["promised_days"]))

In [ ]:
# 1. Engineer features using information available at shipping time.
# Both frames are transformed independently, with no target-derived statistics.
def add_transit_features(data):
    d = data.copy()
    d["ship_date"] = pd.to_datetime(d["ship_date"])
    d["order_date"] = pd.to_datetime(d["order_date"])

    d["order_to_ship_days"] = (d["ship_date"] - d["order_date"]).dt.days
    d["is_weekend"] = d["ship_dow"].isin([5, 6]).astype(int)
    d["is_peak_season"] = d["ship_month"].isin([11, 12]).astype(int)
    d["month_sin"] = np.sin(2 * np.pi * d["ship_month"] / 12)
    d["month_cos"] = np.cos(2 * np.pi * d["ship_month"] / 12)

    d["log_distance"] = np.log1p(d["distance_miles"].clip(lower=0))
    d["miles_per_promised_day"] = d["distance_miles"] / d["promised_days"].clip(lower=1)
    d["log_weight"] = np.log1p(d["weight_lbs"].clip(lower=0))
    d["volume_cuin"] = d["length_in"] * d["width_in"] * d["height_in"]
    d["dim_weight_lbs"] = d["volume_cuin"] / 139.0
    d["billable_weight_lbs"] = d[["weight_lbs", "dim_weight_lbs"]].max(axis=1)
    d["weight_per_pallet"] = d["weight_lbs"] / d["pallet_count"].replace(0, np.nan)

    d["weather_x_congestion"] = d["weather_severity"] * d["carrier_congestion_index"]
    d["distance_x_weather"] = d["distance_miles"] * d["weather_severity"]
    d["peak_x_congestion"] = d["is_peak_season"] * d["carrier_congestion_index"]
    d["carrier_x_service"] = d["carrier"].astype(str) + " | " + d["service_level"].astype(str)
    # Avoid a full lane ID as a one-hot feature until lane support is evaluated.
    return d

tr_fe = add_transit_features(tr)
val_fe = add_transit_features(val)
new_cols = sorted(set(tr_fe.columns) - set(tr.columns))
print("Created", len(new_cols), "new features:")
print(new_cols)

In [ ]:
# 2. Build original vs engineered design matrices.
# Drop target-derived information, identifiers and date objects.
# carrier_cost_usd is billed cost; exclude it for a ship-time prediction model.
excluded = ["transit_days", "delivered_on_time", "shipment_id", "order_date", "ship_date",
            "origin_zip", "dest_zip", "carrier_cost_usd"]

original_features = [c for c in tr.columns if c not in excluded]
engineered_features = [c for c in tr_fe.columns if c not in excluded]

feature_sets = {
    "Original": (tr[original_features].copy(), val[original_features].copy()),
    "Engineered": (tr_fe[engineered_features].copy(), val_fe[engineered_features].copy())
}
y_tr = tr["transit_days"]
y_val = val["transit_days"]
naive_pred = val["promised_days"].to_numpy()

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Keep the preprocessor fitted on training data only.
def make_preprocessor(X):
    cat = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    num = [c for c in X.columns if c not in cat]
    return ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), num),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat)
    ], remainder="drop")

def metrics(y, pred):
    err = np.abs(np.asarray(y) - np.asarray(pred))
    return {"MAE": mean_absolute_error(y, pred),
            "RMSE": np.sqrt(mean_squared_error(y, pred)),
            "Within_1_day_pct": 100 * np.mean(err <= 1)}

print("Promise baseline:", metrics(y_val, naive_pred))

In [ ]:
# 3. Fit the same models on the same chronological validation period.
# Separate estimator objects for every experiment.
def get_models():
    return {
        "Ridge": Ridge(alpha=10.0),
        "Random Forest": RandomForestRegressor(n_estimators=120, max_depth=16,
                                               min_samples_leaf=3, random_state=42, n_jobs=-1),
        "Extra Trees": ExtraTreesRegressor(n_estimators=120, max_depth=16,
                                           min_samples_leaf=3, random_state=42, n_jobs=-1),
        "HistGradientBoosting": HistGradientBoostingRegressor(loss="absolute_error",
                                      max_iter=250, learning_rate=0.05, random_state=42)
    }

results = []
trained = {}
validation_predictions = {}
results.append({"Features": "Baseline", "Model": "Promised days", **metrics(y_val, naive_pred)})

for name, (X_train, X_valid) in feature_sets.items():
    # Boolean missingness flags from the cleaning cell work as numeric features.
    for model_name, estimator in get_models().items():
        pipe = Pipeline([("prep", make_preprocessor(X_train)), ("model", estimator)])
        pipe.fit(X_train, y_tr)
        prediction = pipe.predict(X_valid)
        results.append({"Features": name, "Model": model_name, **metrics(y_val, prediction)})
        trained[(name, model_name)] = pipe
        validation_predictions[(name, model_name)] = prediction
        print(f"{name:10s} | {model_name:22s} | MAE {mean_absolute_error(y_val, prediction):.4f}")

comparison_df = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
display(comparison_df.round(4))

plot_df = comparison_df[comparison_df["Features"] != "Baseline"]
ax = plot_df.pivot(index="Model", columns="Features", values="MAE").plot.bar(figsize=(10, 5))
ax.axhline(metrics(y_val, naive_pred)["MAE"], color="black", linestyle="--", label="Promised days")
ax.set_ylabel("Validation MAE (days)")
ax.set_title("Original versus engineered features")
plt.xticks(rotation=20, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4. Inspect errors for the best candidate (select by validation MAE).
best_row = comparison_df[comparison_df["Features"] != "Baseline"].iloc[0]
best_key = (best_row["Features"], best_row["Model"])
best_prediction = validation_predictions[best_key]
print("Best candidate:", best_key, "MAE:", round(best_row["MAE"], 4))
print("Promised-day MAE:", round(metrics(y_val, naive_pred)["MAE"], 4))

audit = val[["shipment_mode", "carrier", "origin_metro", "dest_metro"]].copy()
audit["actual"] = y_val.to_numpy()
audit["predicted"] = best_prediction
audit["absolute_error"] = abs(audit["actual"] - audit["predicted"])
audit["lane"] = audit["origin_metro"].astype(str) + " -> " + audit["dest_metro"].astype(str)

for group in ["shipment_mode", "carrier", "lane"]:
    grouped = (audit.groupby(group, dropna=False)
                  .agg(shipments=("absolute_error", "size"), MAE=("absolute_error", "mean")))
    if group == "lane":
        grouped = grouped[grouped["shipments"] >= 10]
    print("\nErrors by", group)
    display(grouped.sort_values("MAE", ascending=False).head(15).round(3))

# Permutation importance operates on original named features through the pipeline.
from sklearn.inspection import permutation_importance
X_best_val = feature_sets[best_key[0]][1]
sample_index = X_best_val.sample(n=min(1000, len(X_best_val)), random_state=42).index
importance = permutation_importance(trained[best_key], X_best_val.loc[sample_index],
                                    y_val.loc[sample_index], scoring="neg_mean_absolute_error",
                                    n_repeats=3, random_state=42, n_jobs=1)
importance_df = (pd.DataFrame({"feature": X_best_val.columns,
                               "MAE_increase_if_shuffled": importance.importances_mean})
                   .sort_values("MAE_increase_if_shuffled", ascending=False))
display(importance_df.head(20).round(4))
comparison_df.to_csv("wwex_track_a_model_comparison.csv", index=False)
importance_df.to_csv("wwex_track_a_feature_importance.csv", index=False)

In [ ]:
!pip -q install -U xgboost lightgbm

In [ ]:
import xgboost as xgb
import lightgbm as lgb

from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Training and validation targets
y_train = tr["transit_days"]
y_valid = val["transit_days"]

# Verify feature sets
for name, (X_train, X_valid) in feature_sets.items():

    print(f"\n{name}")

    print("Training shape:", X_train.shape)
    print("Validation shape:", X_valid.shape)

In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=700,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=2.0,
    objective="reg:absoluteerror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_results = {}
xgb_predictions = {}

for feature_name, (X_train, X_valid) in feature_sets.items():

    print(f"\nTraining XGBoost with {feature_name} features")

    pipeline = Pipeline([
        ("preprocessor", make_preprocessor(X_train)),
        ("model", xgb_model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_valid)

    mae = mean_absolute_error(y_valid, predictions)

    rmse = np.sqrt(
        mean_squared_error(y_valid, predictions)
    )

    within_1_day = (
        np.abs(y_valid.to_numpy() - predictions) <= 1
    ).mean() * 100

    xgb_results[feature_name] = {
        "MAE": mae,
        "RMSE": rmse,
        "Within_1_day_pct": within_1_day
    }

    xgb_predictions[feature_name] = predictions

    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Within 1 day: {within_1_day:.2f}%")

In [ ]:
from lightgbm import LGBMRegressor

def prepare_lightgbm_features(X_train, X_valid):

    X_train = X_train.copy()
    X_valid = X_valid.copy()

    categorical_cols = X_train.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in categorical_cols:

        categories = pd.Index(
            X_train[col].dropna().astype(str).unique()
        )

        category_type = pd.CategoricalDtype(
            categories=categories
        )

        X_train[col] = (
            X_train[col].astype("string")
            .astype(category_type)
        )

        X_valid[col] = (
            X_valid[col].astype("string")
            .astype(category_type)
        )

    return X_train, X_valid


lgb_results = {}
lgb_predictions = {}

for feature_name, (X_train, X_valid) in feature_sets.items():

    print(f"\nTraining LightGBM with {feature_name} features")

    X_train_lgb, X_valid_lgb = prepare_lightgbm_features(
        X_train, X_valid
    )

    lgb_model = LGBMRegressor(
        n_estimators=700,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=-1,
        min_child_samples=30,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=2.0,
        objective="regression_l1",
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    lgb_model.fit(
        X_train_lgb,
        y_train,
        categorical_feature="auto"
    )

    predictions = lgb_model.predict(X_valid_lgb)

    mae = mean_absolute_error(
        y_valid, predictions
    )

    rmse = np.sqrt(
        mean_squared_error(y_valid, predictions)
    )

    within_1_day = (
        np.abs(y_valid.to_numpy() - predictions) <= 1
    ).mean() * 100

    lgb_results[feature_name] = {
        "MAE": mae,
        "RMSE": rmse,
        "Within_1_day_pct": within_1_day
    }

    lgb_predictions[feature_name] = predictions

    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Within 1 day: {within_1_day:.2f}%")

In [ ]:
results = []

# Existing best model
results.append({
    "Model": "HistGradientBoosting",
    "Features": "Original",
    "MAE": 0.5446,
    "RMSE": 0.7746,
    "Within_1_day_pct": 86.0085
})

# XGBoost results
for feature_name, metrics_dict in xgb_results.items():

    results.append({
        "Model": "XGBoost",
        "Features": feature_name,
        **metrics_dict
    })

# LightGBM results
for feature_name, metrics_dict in lgb_results.items():

    results.append({
        "Model": "LightGBM",
        "Features": feature_name,
        **metrics_dict
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "MAE",
    ascending=True
)

display(results_df.round(4))

In [ ]:
plt.figure(figsize=(11, 6))

labels = (
    results_df["Model"] +
    " (" +
    results_df["Features"] +
    ")"
)

plt.barh(
    labels,
    results_df["MAE"]
)

plt.axvline(
    0.5446,
    color="red",
    linestyle="--",
    label="Current Best MAE: 0.5446"
)

plt.xlabel("Mean Absolute Error (Days)")
plt.title("WWEX Transit Prediction: Model Comparison")

plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import (
    RandomizedSearchCV,
    TimeSeriesSplit
)

from sklearn.pipeline import Pipeline

from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

RANDOM_STATE = 42

In [ ]:
xgb_params = {

    "model__n_estimators": [
        300, 500, 700, 1000
    ],

    "model__max_depth": [
        3, 4, 5, 6, 7
    ],

    "model__learning_rate": [
        0.01, 0.03, 0.05, 0.07, 0.1
    ],

    "model__min_child_weight": [
        1, 3, 5, 7, 10
    ],

    "model__subsample": [
        0.7, 0.8, 0.9, 1.0
    ],

    "model__colsample_bytree": [
        0.7, 0.8, 0.9, 1.0
    ],

    "model__reg_alpha": [
        0, 0.1, 0.5, 1.0
    ],

    "model__reg_lambda": [
        1, 2, 5, 10
    ]

}

In [ ]:
lgb_params = {

    "model__n_estimators": [
        300, 500, 700, 1000
    ],

    "model__learning_rate": [
        0.01, 0.03, 0.05, 0.07, 0.1
    ],

    "model__num_leaves": [
        7, 15, 31, 63
    ],

    "model__max_depth": [
        3, 5, 7, -1
    ],

    "model__min_child_samples": [
        10, 20, 30, 50
    ],

    "model__subsample": [
        0.7, 0.8, 0.9, 1.0
    ],

    "model__colsample_bytree": [
        0.7, 0.8, 0.9, 1.0
    ],

    "model__reg_lambda": [
        0, 1, 2, 5
    ]

}

In [ ]:
hgb_params = {

    "model__max_iter": [
        200, 300, 500, 700
    ],

    "model__learning_rate": [
        0.01, 0.03, 0.05, 0.1
    ],

    "model__max_leaf_nodes": [
        7, 15, 31, 63
    ],

    "model__max_depth": [
        3, 5, 7, None
    ],

    "model__min_samples_leaf": [
        10, 20, 30, 50
    ],

    "model__l2_regularization": [
        0, 0.1, 1, 5, 10
    ]

}

In [ ]:
tscv = TimeSeriesSplit(
    n_splits=3
)

# Reuse existing feature sets
X_train_xgb, X_val_xgb = feature_sets["Original"]

X_train_lgb, X_val_lgb = feature_sets["Engineered"]

X_train_hgb, X_val_hgb = feature_sets["Original"]

y_train = tr["transit_days"]

y_valid = val["transit_days"]

In [ ]:
def tune_model(
    model,
    param_grid,
    X_train,
    X_valid,
    y_train,
    y_valid,
    model_name,
    n_iter=20
):

    print(f"\n{'='*60}")
    print(f"Tuning {model_name}")
    print(f"{'='*60}")

    start_time = time.time()

    # Build preprocessing + model pipeline
    pipeline = Pipeline([

        (
            "preprocessor",
            make_preprocessor(X_train)
        ),

        (
            "model",
            model
        )

    ])

    # Randomized hyperparameter search
    search = RandomizedSearchCV(

        estimator=pipeline,

        param_distributions=param_grid,

        n_iter=n_iter,

        scoring="neg_mean_absolute_error",

        cv=tscv,

        random_state=RANDOM_STATE,

        n_jobs=2,

        verbose=1,

        refit=True

    )

    # Fit using training data only
    search.fit(
        X_train,
        y_train
    )

    # Best model
    best_model = search.best_estimator_

    # Predict validation data
    predictions = best_model.predict(
        X_valid
    )

    # Calculate performance
    mae = mean_absolute_error(
        y_valid,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_valid,
            predictions
        )
    )

    within_one_day = (
        np.abs(
            np.asarray(y_valid) - predictions
        ) <= 1
    ).mean() * 100

    elapsed = time.time() - start_time

    print("\nBest Parameters:")

    for param, value in search.best_params_.items():
        print(f"{param}: {value}")

    print("\nPerformance:")

    print(
        f"Cross-validation MAE: "
        f"{-search.best_score_:.4f}"
    )

    print(
        f"Validation MAE: {mae:.4f}"
    )

    print(
        f"Validation RMSE: {rmse:.4f}"
    )

    print(
        f"Within 1 day: {within_one_day:.2f}%"
    )

    print(
        f"Training time: {elapsed/60:.2f} minutes"
    )

    return {

        "model": best_model,

        "search": search,

        "predictions": predictions,

        "MAE": mae,

        "RMSE": rmse,

        "Within_1_day_pct": within_one_day,

        "CV_MAE": -search.best_score_,

        "best_params": search.best_params_

    }

In [ ]:
xgb_tuned = tune_model(

    model=XGBRegressor(

        objective="reg:absoluteerror",

        tree_method="hist",

        random_state=42,

        n_jobs=1

    ),

    param_grid=xgb_params,

    X_train=X_train_xgb,

    X_valid=X_val_xgb,

    y_train=y_train,

    y_valid=y_valid,

    model_name="XGBoost",

    n_iter=20

)

In [ ]:
lgb_tuned = tune_model(

    model=LGBMRegressor(

        objective="regression_l1",

        random_state=42,

        n_jobs=1,

        subsample_freq=1,

        verbosity=-1

    ),

    param_grid=lgb_params,

    X_train=X_train_lgb,

    X_valid=X_val_lgb,

    y_train=y_train,

    y_valid=y_valid,

    model_name="LightGBM",

    n_iter=20

)

In [ ]:
hgb_tuned = tune_model(

    model=HistGradientBoostingRegressor(

        loss="absolute_error",

        random_state=42

    ),

    param_grid=hgb_params,

    X_train=X_train_hgb,

    X_valid=X_val_hgb,

    y_train=y_train,

    y_valid=y_valid,

    model_name="HistGradientBoosting",

    n_iter=20

)

In [ ]:
tuned_results = pd.DataFrame([

    {
        "Model": "XGBoost Tuned",
        "MAE": xgb_tuned["MAE"],
        "RMSE": xgb_tuned["RMSE"],
        "Within_1_day_pct": xgb_tuned["Within_1_day_pct"]
    },

    {
        "Model": "LightGBM Tuned",
        "MAE": lgb_tuned["MAE"],
        "RMSE": lgb_tuned["RMSE"],
        "Within_1_day_pct": lgb_tuned["Within_1_day_pct"]
    },

    {
        "Model": "HistGradientBoosting Tuned",
        "MAE": hgb_tuned["MAE"],
        "RMSE": hgb_tuned["RMSE"],
        "Within_1_day_pct": hgb_tuned["Within_1_day_pct"]
    },

    {
        "Model": "XGBoost Previous Best",
        "MAE": 0.5366,
        "RMSE": 0.7582,
        "Within_1_day_pct": 87.6345
    },

    {
        "Model": "LightGBM Previous Best",
        "MAE": 0.5416,
        "RMSE": 0.7680,
        "Within_1_day_pct": 86.9337
    },

    {
        "Model": "HistGradientBoosting Previous Best",
        "MAE": 0.5446,
        "RMSE": 0.7746,
        "Within_1_day_pct": 86.0085
    }

])

tuned_results = tuned_results.sort_values(
    "MAE"
)

display(tuned_results.round(4))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 6))

plt.barh(
    tuned_results["Model"],
    tuned_results["MAE"]
)

plt.axvline(
    0.5366,
    color="red",
    linestyle="--",
    label="Previous Best: 0.5366"
)

plt.xlabel("Mean Absolute Error (Days)")

plt.title(
    "WWEX Transit Prediction: Hyperparameter Tuning"
)

plt.gca().invert_yaxis()

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

#originalfeature set
groups = {
    "promised_days":           ["promised_days"],
    "carrier":                 ["carrier"],
    "service_and_mode":        ["service_level", "shipment_mode"],
    "distance_and_geo":        ["distance_miles", "zone", "origin_lat", "origin_lon", "dest_lat", "dest_lon"],
    "origin_dest_metro":       ["origin_metro", "dest_metro"],
    "package_size_weight":     ["weight_lbs", "length_in", "width_in", "height_in", "pallet_count",
                                "weight_lbs_na", "length_in_na", "width_in_na", "height_in_na"],
    "handling_flags":          ["residential_flag", "signature_required", "declared_value_usd"],
    "weather_fuel_congestion": ["fuel_surcharge_pct", "weather_severity", "carrier_congestion_index"],
    "ship_day_and_month":      ["ship_dow", "ship_month"],
}
base_cols = [c for g in groups.values() for c in g]
assert set(base_cols) == set(original_features), "groups must match the Original feature set"

#candidate groups to ADD
add_ons = {
    "carrier_cost_usd":   ["carrier_cost_usd"],
    "eng_calendar":       ["order_to_ship_days", "is_weekend", "is_peak_season", "month_sin", "month_cos"],
    "eng_size_transforms": ["log_distance", "log_weight", "volume_cuin", "dim_weight_lbs",
                            "billable_weight_lbs", "weight_per_pallet"],
    "eng_interactions":   ["weather_x_congestion", "distance_x_weather", "peak_x_congestion", "carrier_x_service"],
    "eng_promise_ratio":  ["miles_per_promised_day"],
}

#time-ordered cross validation inside the training data
assert tr["ship_date"].is_monotonic_increasing  #TimeSeriesSplit needs rows in time order
cv_splitter = TimeSeriesSplit(n_splits=3)
y_tr, y_val = tr["transit_days"], val["transit_days"]

def make_xgb(seed=42, fast=True):
    return XGBRegressor(n_estimators=250 if fast else 700, learning_rate=0.08 if fast else 0.03,
                        max_depth=5, min_child_weight=5, subsample=0.85, colsample_bytree=0.85,
                        reg_alpha=0.1, reg_lambda=2.0, objective="reg:absoluteerror",
                        tree_method="hist", random_state=seed, n_jobs=-1)

def evaluate(cols, seed=42):
    """Returns (time-ordered CV MAE on train, validation MAE) for one feature list."""
    cv = []
    for fit_idx, test_idx in cv_splitter.split(tr_fe):
        Xa, Xb = tr_fe.iloc[fit_idx][cols], tr_fe.iloc[test_idx][cols]
        pipe = Pipeline([("prep", make_preprocessor(Xa)), ("model", make_xgb(seed))]).fit(Xa, y_tr.iloc[fit_idx])
        cv.append(mean_absolute_error(y_tr.iloc[test_idx], pipe.predict(Xb)))
    pipe = Pipeline([("prep", make_preprocessor(tr_fe[cols])), ("model", make_xgb(seed))]).fit(tr_fe[cols], y_tr)
    return float(np.mean(cv)), mean_absolute_error(y_val, pipe.predict(val_fe[cols]))

#how much does MAE move just from random seeds?
seed_runs = pd.DataFrame([evaluate(base_cols, s) for s in [1, 2, 3, 4, 5]], columns=["cv_MAE", "val_MAE"])
noise_floor = max(0.002, 2 * seed_runs["cv_MAE"].std())
print(seed_runs.round(4))
print(f"\nNoise floor: {noise_floor:.4f} days. Differences smaller than this are treated as ties.")

In [ ]:
#building the candidate feature sets
candidates = {"Base (all original features)": base_cols}
for g in groups:                                           # remove one group at a time
    candidates[f"Drop {g}"] = [c for c in base_cols if c not in groups[g]]
for g in add_ons:                                          # add one group at a time
    candidates[f"Add {g}"] = base_cols + add_ons[g]
all_eng = [c for k, v in add_ons.items() if k != "carrier_cost_usd" for c in v]
candidates["Add all engineered (no cost)"] = base_cols + all_eng

rows = []
for name, cols in candidates.items():
    cv_mae, val_mae = evaluate(cols)
    rows.append({"feature_set": name, "n_features": len(cols), "cv_MAE": cv_mae, "val_MAE": val_mae})
exp_results = pd.DataFrame(rows).set_index("feature_set")

base_cv = exp_results.loc["Base (all original features)", "cv_MAE"]
exp_results["cv_change_vs_base"] = exp_results["cv_MAE"] - base_cv      # negative = better than base
display(exp_results.sort_values("cv_MAE").round(4))

In [ ]:
#A group is "not needed" if removing it does not hurt (within the noise floor).
#A group "helps" if adding it improves CV MAE by more than the noise floor.
not_needed = [g for g in groups  if exp_results.loc[f"Drop {g}", "cv_MAE"] <= base_cv + noise_floor]
needed     = [g for g in groups  if g not in not_needed]
helpful    = [g for g in add_ons if exp_results.loc[f"Add {g}", "cv_MAE"] <= base_cv - noise_floor]
print("Needed (removing them hurts):    ", needed)
print("Not needed (removing is a tie):  ", not_needed)
print("Helpful to add:                  ", helpful)

#combining what the tests say and test the combination too
combined_cols = [c for g in needed for c in groups[g]] + [c for g in helpful for c in add_ons[g]]
candidates["Combined (needed groups + helpful additions)"] = combined_cols
cv_mae, val_mae = evaluate(combined_cols)
exp_results.loc["Combined (needed groups + helpful additions)"] = [len(combined_cols), cv_mae, val_mae, cv_mae - base_cv]
print("\nAll feature sets, best CV first (the combined set is the last one we tested):")
display(exp_results.sort_values("cv_MAE").round(4))

#picking the final set with the best CV score
best_cv = exp_results["cv_MAE"].min()
tied = exp_results[exp_results["cv_MAE"] <= best_cv + noise_floor]
final_name = tied.sort_values(["n_features", "cv_MAE"]).index[0]
final_cols = candidates[final_name]
print(f"\nBest CV MAE: {best_cv:.4f} | sets within the noise floor of it: {len(tied)}")
print(f"FINAL FEATURE SET: {final_name} ({len(final_cols)} features)")
print("Uses carrier_cost_usd:", "carrier_cost_usd" in final_cols)

In [ ]:
# refit the final set with the full model settings and check it once on validation
final_pipe = Pipeline([("prep", make_preprocessor(tr_fe[final_cols])),
                       ("model", make_xgb(42, fast=False))]).fit(tr_fe[final_cols], y_tr)
final_pred = final_pipe.predict(val_fe[final_cols])

def report(pred):
    return {"MAE": mean_absolute_error(y_val, pred),
            "RMSE": mean_squared_error(y_val, pred) ** 0.5,
            "Within_1_day_pct": 100 * (np.abs(y_val.to_numpy() - pred) <= 1).mean()}

final_report = pd.DataFrame({
    "Promised days (naive)": report(val_fe["promised_days"].to_numpy()),
    "XGBoost, Original features (previous best)": report(xgb_predictions["Original"]),
    f"XGBoost, final set: {final_name}": report(final_pred),
}).T
display(final_report.round(4))

In [ ]:
# which hich ones actually carry weight in the final model?
from sklearn.inspection import permutation_importance
Xv = val_fe[final_cols]
sample_idx = Xv.sample(n=min(3000, len(Xv)), random_state=42).index
pi = permutation_importance(final_pipe, Xv.loc[sample_idx], y_val.loc[sample_idx],
                            scoring="neg_mean_absolute_error", n_repeats=3, random_state=42, n_jobs=1)
feature_importance = (pd.DataFrame({"feature": final_cols, "MAE_increase_if_shuffled": pi.importances_mean})
                        .sort_values("MAE_increase_if_shuffled", ascending=False).reset_index(drop=True))
display(feature_importance.round(4))
print("Features that barely matter (shuffling them changes MAE by < 0.001):",
      feature_importance.loc[feature_importance["MAE_increase_if_shuffled"] < 0.001, "feature"].tolist())

In [ ]:
#one row per validation shipment, with the model's miss attached
best_pred = xgb_predictions["Original"]

audit = val[["shipment_mode", "carrier", "service_level", "origin_metro", "dest_metro",
             "promised_days", "ship_date"]].copy()
audit["actual"] = y_valid.to_numpy()
audit["predicted"] = best_pred
audit["abs_err"] = (audit["actual"] - audit["predicted"]).abs()
audit["signed_err"] = audit["predicted"] - audit["actual"]          # negative = predicted too fast
audit["naive_abs_err"] = (audit["actual"] - audit["promised_days"]).abs()
audit["within1"] = (audit["abs_err"] <= 1).astype(int)
audit["lane"] = audit["origin_metro"].astype(str) + " -> " + audit["dest_metro"].astype(str)

def summarize(g):
    out = g.agg(n=("abs_err", "size"),
                model_MAE=("abs_err", "mean"),
                naive_MAE=("naive_abs_err", "mean"),
                bias=("signed_err", "mean"),
                within1=("within1", "mean"),
                p90_err=("abs_err", lambda s: s.quantile(0.9)),
                err_std=("abs_err", "std"))
    out["gain_vs_naive"] = out["naive_MAE"] - out["model_MAE"]        # how much better than the promise
    out["std_error"] = out["err_std"] / np.sqrt(out["n"])             # how noisy this MAE is
    return out.drop(columns="err_std")

print("Overall model MAE:", round(audit["abs_err"].mean(), 4),
      "| promised-days MAE:", round(audit["naive_abs_err"].mean(), 4),
      "| validation shipments:", len(audit))

In [ ]:
#by shipment mode, service level, and carrier
for col in ["shipment_mode", "service_level", "carrier"]:
    print(f"\n=== Errors by {col} ===")
    display(summarize(audit.groupby(col)).sort_values("model_MAE", ascending=False).round(3))

print("\n=== Carrier within shipment mode ===")
display(summarize(audit.groupby(["shipment_mode", "carrier"])).sort_values("model_MAE", ascending=False).round(3))

In [ ]:
#by lane, origin to destination
MIN_N = 30   # lanes with fewer validation shipments are too noisy to rank
lane_stats = summarize(audit.groupby("lane"))
solid = lane_stats[lane_stats["n"] >= MIN_N]
print(f"{len(lane_stats)} lanes in validation | {len(solid)} have n >= {MIN_N} | median lane size = {lane_stats['n'].median():.0f}")

print(f"\n=== Weakest lanes (n >= {MIN_N}) ===")
display(solid.sort_values("model_MAE", ascending=False).head(10).round(3))

print(f"\n=== Strongest lanes (n >= {MIN_N}) ===")
display(solid.sort_values("model_MAE").head(5).round(3))

print("\n=== Lanes where the model barely beats the carrier promise (gain < 0.1 day) ===")
display(solid[solid["gain_vs_naive"] < 0.1].sort_values("gain_vs_naive").round(3))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))

# carrier: model vs. the carrier's own promise
c = summarize(audit.groupby("carrier")).sort_values("model_MAE")
x = np.arange(len(c)); w = 0.38
ax[0].bar(x - w/2, c["model_MAE"], w, label="Model")
ax[0].bar(x + w/2, c["naive_MAE"], w, label="Promised days")
ax[0].set_xticks(x); ax[0].set_xticklabels(c.index, rotation=30, ha="right")
ax[0].set_ylabel("MAE (days)"); ax[0].set_title("MAE by carrier: model vs. promise"); ax[0].legend()

# lane MAE vs. lane size (small lanes scatter wildly, which is noise, not signal)
ax[1].scatter(lane_stats["n"], lane_stats["model_MAE"], alpha=0.6)
ax[1].axhline(audit["abs_err"].mean(), color="red", linestyle="--", label="Overall MAE")
ax[1].axvline(MIN_N, color="gray", linestyle=":", label=f"n = {MIN_N}")
ax[1].set_xlabel("Validation shipments in lane"); ax[1].set_ylabel("Lane MAE (days)")
ax[1].set_title("Lane MAE vs. lane size"); ax[1].legend()

# signed error by what actually happened (below 0 = predicted too fast)
t = audit[audit["actual"] <= 12].groupby("actual")["signed_err"].mean()
ax[2].bar(t.index, t.values); ax[2].axhline(0, color="black", linewidth=0.8)
ax[2].set_xlabel("Actual transit days"); ax[2].set_ylabel("Mean (predicted - actual), days")
ax[2].set_title("Bias by actual transit time")

plt.tight_layout(); plt.show()

In [ ]:
big = audit[audit["abs_err"] > 2]
print(f"Misses over 2 days: {len(big)} of {len(audit)} ({len(big)/len(audit):.1%}) | "
      f"predicted too fast in {(big['signed_err'] < 0).mean():.0%} of them")
display(big.groupby("carrier").size().rename("big_misses").to_frame()
           .assign(share=lambda d: d["big_misses"] / d["big_misses"].sum())
           .sort_values("big_misses", ascending=False).round(3))

audit["month"] = audit["ship_date"].dt.to_period("M").astype(str)
print("\n=== Error by ship month (is the model drifting?) ===")
display(summarize(audit.groupby("month"))[["n", "model_MAE", "naive_MAE", "bias"]].round(3))

In [ ]:
#is Falcon Post hard to predict everywhere, or only on a few lanes?
f = audit[audit["carrier"] == "Falcon Post"]
o = audit[audit["carrier"] != "Falcon Post"]
for name, d in [("Falcon Post", f), ("Other carriers", o)]:
    print(f"{name:15s} n={len(d):5d} | MAE={d['abs_err'].mean():.3f} | bias={d['signed_err'].mean():+.3f} | "
          f"off by more than 1 day: {(d['abs_err'] > 1).mean():.1%}")

print("\n=== Falcon Post by service level ===")
display(summarize(f.groupby("service_level")).round(3))

# same lane, different carrier
#is Falcon Post worse on the lanes both serve?
by_lane = (f.groupby("lane")["abs_err"].agg(falcon_n="size", falcon_MAE="mean")
             .join(o.groupby("lane")["abs_err"].agg(other_n="size", other_MAE="mean"), how="inner"))
by_lane = by_lane[(by_lane["falcon_n"] >= 10) & (by_lane["other_n"] >= 10)]
by_lane["gap"] = by_lane["falcon_MAE"] - by_lane["other_MAE"]
print(f"\nLanes with 10+ shipments for both: {len(by_lane)} | Falcon Post worse in {(by_lane['gap'] > 0).sum()} of them "
      f"| median gap {by_lane['gap'].median():.3f} days")
display(by_lane.sort_values("gap", ascending=False).head(10).round(3))

In [ ]:
audit["final_pred"] = final_pred
audit["final_abs_err"] = (audit["actual"] - audit["final_pred"]).abs()

def compare(col, min_n=1):
    g = audit.groupby(col).agg(n=("abs_err", "size"), previous_MAE=("abs_err", "mean"),
                               final_MAE=("final_abs_err", "mean"), promise_MAE=("naive_abs_err", "mean"))
    g["change"] = g["final_MAE"] - g["previous_MAE"]
    return g[g["n"] >= min_n]

#is the overall difference bigger than chance? (paired comparison, same shipments)
diff = audit["abs_err"] - audit["final_abs_err"] # positive = final set was closer
se = diff.std() / np.sqrt(len(diff))
print(f"Overall: previous MAE {audit['abs_err'].mean():.4f} -> final MAE {audit['final_abs_err'].mean():.4f} "
      f"| average improvement {diff.mean():+.4f} days (+/- {1.96 * se:.4f}, 95% range)")

for col in ["shipment_mode", "service_level", "carrier"]:
    print(f"\n=== {col}: previous vs. final ===")
    display(compare(col).sort_values("change").round(3))

lane_cmp = compare("lane", min_n=MIN_N)
print(f"\n=== lanes with {MIN_N}+ shipments: improved in {(lane_cmp['change'] < 0).sum()} of {len(lane_cmp)} ===")
display(lane_cmp.sort_values("change").head(5).round(3))
display(lane_cmp.sort_values("change").tail(5).round(3))

In [ ]:
#Step 1. Check shipment categories, this confirms how many Parcel and LTL shipments we have.
print(tr["shipment_mode"].value_counts())
print(val["shipment_mode"].value_counts())

In [ ]:
#Step 2. Train separate models
#What happens?
# - First, XGBoost learns only from Parcel shipments.
# - Then, XGBoost learns only from LTL shipments.
# - Each model makes predictions for its own shipment type.
# - We calculate the MAE for each.

from xgboost import XGBRegressor

separate_results = {}
separate_predictions = {}

for mode in ["parcel", "ltl_freight"]:

    # Select only this shipment type
    train_mask = tr["shipment_mode"] == mode
    val_mask = val["shipment_mode"] == mode

    X_train = tr_fe.loc[train_mask, engineered_features]
    X_valid = val_fe.loc[val_mask, engineered_features]

    y_train_mode = tr.loc[train_mask, "transit_days"]
    y_valid_mode = val.loc[val_mask, "transit_days"]

    # Create a fresh XGBoost model
    model = XGBRegressor(
        n_estimators=700,
        learning_rate=0.03,
        max_depth=5,
        min_child_weight=5,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=2.0,
        objective="reg:absoluteerror",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    # Preprocess and train
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor(X_train)),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train_mode)

    # Predict
    predictions = pipeline.predict(X_valid)

    # Calculate MAE
    mae = mean_absolute_error(y_valid_mode, predictions)

    separate_results[mode] = mae
    separate_predictions[mode] = predictions

    print(f"{mode} MAE: {mae:.4f}")

In [ ]:
#Step 3. Compare against combined model
comparison = pd.DataFrame({
    "Model": [
        "Combined Model",
        "Parcel Model",
        "LTL Model"
    ],
    "MAE": [
        xgb_results["Engineered"]["MAE"],
        separate_results["parcel"],
        separate_results["ltl_freight"]
    ]
})

display(comparison.round(4))

In [ ]:
#Fair comparison
combined_predictions = xgb_predictions["Engineered"]

fair_results = []

for mode in ["parcel", "ltl_freight"]:

    mask = val["shipment_mode"] == mode

    # Combined model on this shipment type
    combined_mae = mean_absolute_error(
        y_val[mask],
        combined_predictions[mask]
    )

    # Separate model on this shipment type
    separate_mae = separate_results[mode]

    fair_results.append({
        "Shipment Mode": mode,
        "Combined MAE": combined_mae,
        "Separate MAE": separate_mae,
        "Improvement": combined_mae - separate_mae
    })

display(pd.DataFrame(fair_results).round(4))